# PDF Data Extraction with Open-Weight Models

**2026 UIUC Workshop -- de Medeiros Insect Flower Visitors Lab, Field Museum**

This is the 2026 rewrite of the 2025 ESA workshop notebook
(`pdf_data_extraction.ipynb`, Claude Haiku 4.5 via AWS Bedrock). Everything
here runs on **open-weight models**, **free on Google Colab**, with **no API
key to distribute**.

Audience: systematic entomologists, not software engineers. No programming
background beyond basic Python is assumed -- if a cell looks unfamiliar,
treat that as a cue to ask, not a sign that you are behind.

Repository: `de-Medeiros-insect-lab/2026_UIUC_workshop_llm_pdf_extraction`


## 0. Why this changed since 2025

Three things are different this year, and each one removes or replaces a
whole section of the old notebook.

1. **No key distribution.** 2025 ran Claude through AWS Bedrock on the
   instructor's credits, and every student needed an individually issued
   key. Open models on Ollama need no key at all -- that deletes the entire
   setup ceremony, and the anxiety about accidentally spending someone
   else's grant money.
2. **Structured output is enforced, not negotiated.** 2025 spent several
   cells coaxing Claude into valid JSON, then repairing whatever came back.
   Ollama's `format=` parameter constrains decoding to a JSON Schema, so
   non-conforming output is not merely discouraged -- it is
   unrepresentable. Later today, Section 6 replaces prompt-wrangling with
   schema *design*.
3. **PDFs stop being free.** Claude read PDFs natively; the models we use
   today do not. We have to confront directly that a born-digital paper has
   a text layer, while a scan is only pixels until something reads it.
   That distinction turns out to be the most useful thing in this
   notebook, and Sections 3 and 4 are built entirely around it.

There is also a case for running models locally at all, beyond dodging API
keys: you can archive open weights alongside your data when you publish a
methods section, which you cannot do with a commercial API endpoint that may
not exist in five years. Unpublished specimen records and localities for
endangered taxa also never have to leave your machine.


## 1. Ollama on Colab

We will run everything through [Ollama](https://ollama.com), a small server
that manages open-weight models on your own machine (or, today, your Colab
VM) and exposes them through a simple API. Two models, one job each:

| model | role |
| --- | --- |
| `qwen3.5:9b` | reasoning, extraction, tool use -- the model you will talk to all day |
| `deepseek-ocr` | a dedicated transcription model for reading scanned pages |

The setup cell below installs Ollama and starts its server *only* when
running on Colab; locally it is a no-op, because you already have both. Run
it either way and keep reading -- on a fresh Colab VM the first run downloads
real weights and takes a couple of minutes.


In [ ]:
# Colab setup. On a fresh VM this takes ~2 minutes; keep reading while it runs.
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !curl -fsSL https://ollama.com/install.sh | sh
    !pip -q install ollama pymupdf pydantic pandas
    !git clone -q https://github.com/de-Medeiros-insect-lab/2026_UIUC_workshop_llm_pdf_extraction.git
    os.chdir("2026_UIUC_workshop_llm_pdf_extraction")
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

from workshop_lib import server_ready, CHAT_MODEL, OCR_MODEL
assert server_ready(120), "Ollama did not start"
print("Ollama is up")


### Check for a GPU

These models are small by LLM standards, but still far too slow on a CPU for
a live workshop. Run this now, before we go any further.


In [ ]:
import subprocess

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True)
    have_gpu = gpu.returncode == 0
except FileNotFoundError:
    # No nvidia-smi binary at all -- e.g. a Mac, or a Colab CPU-only runtime.
    have_gpu = False

if have_gpu:
    print("GPU:", gpu.stdout.strip())
else:
    print("NO GPU on this runtime.")
    print("Colab often refuses free GPUs at busy times of day.")
    print("Please pair up with a neighbour who has one -- the models are far")
    print("too slow on CPU for a workshop.")


### Pull the models

About 14 GB combined. Start this now -- we will keep talking while it
downloads in the background.


In [ ]:
# ~14 GB total. Start this now; we will talk while it downloads.
!ollama pull {CHAT_MODEL}
!ollama pull {OCR_MODEL}
!ollama list


## 2. Messages, system prompts, and role-play

Chat models are driven by a list of *messages*, each tagged with a role:
`system` (instructions that set the model's persona and constraints),
`user` (what you ask), and `assistant` (what the model answers). The
`system` message is not decoration -- it is the single strongest lever you
have over the model's behaviour, and it works because a chat model is
fundamentally playing a role you assign it:

> Shanahan, M., McDonell, K. & Reynolds, L. *Role play with large language
> models.* Nature 623, 493-498 (2023).

Their argument, in short: a chat model does not "become" an expert
coleopterist because you told it so -- it predicts what an expert
coleopterist's reply would look like, and a good system prompt is good stage
direction. The effect below is not a party trick; it is the mechanism.


In [ ]:
import ollama

reply = ollama.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "system",
         "content": "You are an expert coleopterist. Answer in two sentences."},
        {"role": "user",
         "content": "What is a rostrum, and which beetles have one?"},
    ],
    think=False,
    options={"temperature": 0},
)
print(reply.message.content)


**Before moving on, check that answer against what you actually know.**

When this notebook was written, the model answered by saying the rostrum
"characterizes the superfamily Staphylinoidea (including weevils...)" --
weevils are Curculionoidea, not Staphylinoidea (that's the rove beetles). A
confident, fluent, flatly wrong statement about the one subject this room is
full of world experts in -- and worth treating as representative, not as a
freak one-off.

Model output can change between runs, and will certainly change as the model
itself is updated, so the answer you actually got above may read differently.
Read it anyway, and check it the same way you would check a student's answer:
is this actually right, given what you already know?

The point does not depend on the specific wording: a 9B model is fluent
about taxonomy and not reliable about it. Fluency and accuracy are separate
properties, and a model can have plenty of one with none of the other. That
is exactly why the rest of this workshop does not ask the model what it
knows about a specimen -- it extracts text *from* a document you can point
to and check. It is also why Section 6's schema requires a `source_text`
field on every extracted trait: an answer you cannot trace back to a page is
not evidence, however fluently it is stated.


### Hands-on 1

Write your own system prompt. Pick a persona and a question -- a curator
sorting a drawer of unidentified weevils, a strict referee checking a species
description, a museum docent explaining a specimen to visitors, whatever you
like -- and see how far the answer moves.


In [ ]:
reply = ollama.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "WRITE A ROLE FOR THE MODEL HERE"},
        {"role": "user",   "content": "ASK YOUR QUESTION HERE"},
    ],
    think=False,
    options={"temperature": 0},
)
print(reply.message.content)


**Now run the same question twice more:** edit and re-run the Hands-on 1 cell above -- once with `"content": ""` for the system message (no persona at all), and once with a long, detailed persona in its place. Compare the three answers you get. Hosted models like the one used in 2025 already answer reasonably with no system prompt; a 9B model like this one needs much firmer steering, and the gap between "no persona" and "detailed persona" is larger than you might expect.


## 3. PDFs are not text: text layer vs. pixels

A PDF page is not "text" -- it is a page of ink positions, and text only
comes along for the ride when the file happens to carry an embedded text
layer. Two very different documents sit in `example_pdfs/`:

- **`deMedeiros2013Zootaxa.pdf`** -- born-digital, 2013, three
  *Anchylorhynchus* weevil species. Its text layer is exact.
- **`Marshall1929_AnnMagNatHist.pdf`** -- an 8-page scan of a 1929 paper. It
  *also* has a text layer (someone ran OCR on it at some point), but that
  layer can be silently wrong.

`workshop_lib.open_pdf` and `get_page_text` (pages are 1-based, like a real
paginated document) get you the text layer for free -- when it is
trustworthy.


In [ ]:
from workshop_lib import open_pdf, get_page_text, render_page
from IPython.display import Image, display
import base64

modern = open_pdf("example_pdfs/deMedeiros2013Zootaxa.pdf")
legacy = open_pdf("example_pdfs/Marshall1929_AnnMagNatHist.pdf")

print("MODERN, born-digital:")
print(repr(get_page_text(modern, 1)[:200]))
print("\nLEGACY, a 1929 scan -- note this is NOT empty:")
print(repr(get_page_text(legacy, 5)[:200]))


The legacy page's text is not empty, and it reads as plausible prose -- that
is exactly what makes silently-corrupt OCR the dangerous case rather than the
obvious one. Look at the page image itself and judge for yourself:


In [ ]:
display(Image(data=base64.b64decode(render_page(legacy, 5, dpi=100))))


The text layer said `Cureulionidse`; the page plainly reads `Curculionidae`.
Nothing raised an exception and no field came back empty -- an extraction
pipeline trusting the text layer alone would confidently attribute this
species to a family that matches no taxonomic authority anywhere. Later
today, Section 4 teaches the model to read pages like this one from their
pixels instead of trusting the text layer.


## 4. OCR: reading pixels when the text layer lies

Section 3 showed that the legacy scan's text layer is not empty and not
obviously broken -- it reads as plausible prose while silently saying
`Cureulionidse` for `Curculionidae`. The fix is to stop trusting the text
layer for this document and read the page image itself with a model built
for that job.

`workshop_lib.ocr_page` renders the page to an image (the same
`render_page` you already used to look at it) and sends it to
`deepseek-ocr`, a small vision model trained specifically to transcribe --
not summarise, not interpret, just copy what is printed. Compare its
transcription of page 5 against the text layer you already have:

In [ ]:
from workshop_lib import ocr_page

dirty = get_page_text(legacy, 5)
clean = ocr_page(legacy, 5)

for label, text in [("TEXT LAYER", dirty), ("DEEPSEEK-OCR", clean)]:
    print(f"--- {label} ---")
    print(" ".join(text.split())[:180])
    print()

**This was measured, not eyeballed as "looks better."** Twenty taxonomic
terms on page 5 were checked by eye against each transcription: the PDF's
own text layer got 16 of 20 right, `deepseek-ocr` got 19 of 20, and
`qwen3.5:9b` -- the general chat model, pointed at the same page image --
also got 19 of 20. So a general-purpose reasoning model can do this job
about as accurately as a dedicated OCR model. The reason to use
`deepseek-ocr` anyway is cost, and it is a stark difference:

| dpi | `qwen3.5:9b` prompt tokens | `deepseek-ocr` prompt tokens |
| --- | --- | --- |
| 72  | 271  | 331 |
| 100 | 475  | 961 |
| 150 | 1043 | 961 |

The general model's prompt-token cost grows with image resolution at every
step, the way you would expect a vision encoder to behave. `deepseek-ocr`'s
grows too at first, then goes flat between 100 and 150 dpi -- above some
threshold it compresses a page into a fixed token budget instead of scaling
with pixel count the way a general vision encoder does. Per page, warm,
that difference plays out as 6.4 s for `deepseek-ocr` against 24.7 s for
`qwen3.5:9b`: roughly 4x faster for equivalent accuracy. That is what a
purpose-built OCR model buys you -- not better answers, cheaper and faster
ones at the same quality.

## 5. Thinking

`qwen3.5:9b` is a *reasoning* model: pass `think=True` and it produces a
chain of thought (`reply.message.thinking`), reported separately from its
final answer (`reply.message.content`). Here it works through a short
arithmetic problem out loud before answering:


In [ ]:
reply = ollama.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user",
               "content": "A beetle is 4.2 mm long and 1.4 mm wide. "
                          "What is the length-to-width ratio?"}],
    think=True,
    options={"temperature": 0},
)
print("--- reasoning ---");  print(reply.message.thinking)
print("--- answer ---");     print(reply.message.content)


**This setting is not free, and it is not always a win.** Passing
`think=True` on a *transcription* task -- read this page and copy the text --
made this same model emit 123,055 characters of reasoning, run into its own
output-length limit, and return no answer at all. Transcription is recall:
there is nothing on the page to reason *about*, so reasoning has nothing to
do except ruminate. That is why every transcription and one-shot extraction
call in this notebook passes `think=False`.

Hold onto that thought -- it is not the end of the story. Section 7 puts this
same model in charge of *deciding* whether a page's text can be trusted, and
there the setting flips: judging trustworthiness is reasoning work, and the
loop does not function without `think=True`. The transferable skill is not
"always start with this setting off" -- it is recognising which kind of task
is in front of you.


## 6. Structured output: schemas, not hope

Section 0 promised that structured output would stop being negotiated. Here
is what "negotiated" looked like in the 2025 version of this workshop: ask
the model for JSON in plain English, get back something that is *usually*
JSON, and write code to cope with the times it is not. That version spent
four cells escalating the ask -- add an example, strengthen the system
prompt, wrap the instruction in XML tags -- and still finished by running
the result through `json-repair` to patch whatever came back. Watch the
failure mode it was built to survive (run it a couple of times; nothing
here guarantees the same shape twice):

In [ ]:
# The 2025 approach: ask nicely and hope. Run it a couple of times.
reply = ollama.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user",
               "content": "List two traits of this beetle as JSON:\n\n"
                          + clean[:1500]}],
    think=False, options={"temperature": 0},
)
print(reply.message.content[:400])

Whatever came back above, `reply.message.content` is not guaranteed to be
JSON at all: it may be wrapped in ```` ```json ```` fences that a Markdown
renderer strips but `json.loads` will not, it may use different field names
on a different run, and nothing stops the model from inventing a field,
omitting one, or answering in prose instead. That gap is exactly what
`json-repair` existed to paper over: strip the fences, fix a trailing comma,
hope the shape is close enough to salvage.

Ollama's `format=` parameter removes the gap instead of patching it after
the fact. Give it a JSON Schema -- which `extract()` builds automatically
from the `Extraction` pydantic model below -- and the schema is enforced
*during decoding*, token by token. The model is not being asked nicely to
produce conforming output; output that does not conform cannot be generated
in the first place. `json-repair` is still worth knowing about for any
provider that lacks this feature, but you no longer need it here. Start with the case this pipeline is
actually built for -- a born-digital page with real binomial names and
real measurements:

In [ ]:
from workshop_lib import extract, to_dataframe

species_text = (get_page_text(modern, 2) + get_page_text(modern, 4)
                + get_page_text(modern, 6))
result = extract(species_text)
df = to_dataframe(result)
print(f"{df['species'].nunique()} species, {len(df)} trait rows")
df.groupby("species").head(3)

When this notebook was written, this produced three species --
`Anchylorhynchus pinocchio`, `A. centrosquamatus`, `A. luteobrunneus` -- and
104 trait rows in total (35, 36, and 33 respectively), every one traceable
through `source_text` back to the sentence it came from. This is Section
6's actual payoff: schema-enforced extraction against the kind of text this
pipeline is built for -- clean, born-digital, proper binomial names, real
measurements -- just works, and hands you a dataframe you could paste into a
spreadsheet.

Now the contrast. Page 5 of the legacy scan is not a species description at
all -- it is the middle of a taxonomic key, and that difference matters more
than the OCR quality does:

In [ ]:
from pydantic import ValidationError

try:
    legacy_result = extract(clean)
    legacy_df = to_dataframe(legacy_result)
except ValidationError as e:
    legacy_df = None
    print("Extraction rejected:\n")
    print(e)

if legacy_df is not None:
    display(legacy_df)

If that raised, it did so for a real reason, not a staged one: page 5 sits
in the middle of a taxonomic key, where a couplet can name a genus without
yet attaching a species epithet. `Species` requires a binomial name on
purpose (see its docstring in `workshop_lib.py`) -- a genus-only extraction
usually means the model has not actually found the species being described,
and silently accepting it would hide that failure rather than surface it.

When a schema rejects real output like this, there are three honest
responses, and which one is right depends on the corpus, not on the tool:

1. **Loosen the schema.** If genus-only records are legitimate data for what
   you are building -- a faunal checklist that deliberately includes
   genus-level records, say -- relax `Species._binomial` to accept them, and
   add a field flagging the record as genus- rather than species-level, so
   nothing is silently downgraded to look like a full identification.
2. **Improve the prompt.** Tell the model explicitly to skip any entry that
   lacks a full binomial rather than reporting the nearest name it can find.
   This helps when the problem is what the model chooses to report, not
   what the schema is willing to accept.
3. **Accept the rejection.** Let extraction fail loudly on this page and
   move on -- the default here, and the one this notebook picks.

For the Marshall paper specifically, option 3 is the right call: a
taxonomic key is not a species description, this pipeline exists to extract
species accounts, and a schema loose enough to accept a genus-only
"species" here would also quietly accept it when a model simply fails to
find the binomial in a real description elsewhere in the same document. A
loud, page-5-shaped failure is more useful than a row in `df` that looks
legitimate and is not.

### Hands-on 2

Edit the cell above and try it on different text. `dirty` (the legacy
page's *corrupt* text layer, instead of the OCR'd `clean`) is a good first
try -- does the corruption change *which* check fails, or does it fail the
same way? `get_page_text(modern, 3)` (the "Remarks" section for A.
pinocchio -- prose about the species, but no explicit measurements) is
worth trying too, to see how sparse the trait list gets when the source
text doesn't actually describe measurable traits. Either way, the schema
guarantees the *shape* of whatever comes back; it has no opinion about
whether the content is correct, and it will happily validate a well-formed
wrong answer. That is still your job to check -- which is exactly why
`source_text` is a required field on every trait: every row should be
traceable back to the page it came from.

The schema does catch some content errors, though -- specifically the ones
you told it to care about. `Trait` carries a validator that rejects
implausible millimetre measurements (nothing in this literature is 5 metres
long), the same mechanism that just rejected a genus-only species name
above. Trigger it directly:

In [ ]:
from workshop_lib import Trait
from pydantic import ValidationError

try:
    Trait(anatomical_part="body", trait="length", value="5000",
          units="mm", source_text="body 5000 mm long")
except ValidationError as e:
    print("Rejected, as it should be:\n", e)

A schema makes the *shape* of a mistake impossible; it cannot make the
*model* right, and -- as page 5 just demonstrated -- it cannot even
guarantee the model found what you actually asked for. Section 7 turns to a
different kind of mistake: not a malformed answer, but a confident,
well-formed, wrong one, arrived at by trusting a corrupt page. This time
the question is whether the model can be put in charge of catching it
itself.

## 7. Putting the model in charge of the choice

The ramp so far: Section 3 read a modern, born-digital PDF where the text
layer simply worked. Section 4 OCR'd the 1929 scan *by hand* -- you already
knew page 5's text layer was suspect, so you called `ocr_page` yourself.

Real collections do not arrive pre-sorted into "trust this" and "don't"
piles. What if the model decided, page by page, which tool to reach for?
`TOOL_SCHEMAS` in `workshop_lib.py` already describes both `get_page_text`
and `ocr_page` to the model as callable tools, and `run_tool_loop` drives
the back-and-forth: the model calls a tool, reads the result, and decides
whether to call another. Handing over that decision is only safe here
*because* the last two sections taught you exactly what there is to check
-- you are not delegating blindly, you are delegating a judgement you
already know how to make yourself.

First, the failure that motivates the setting you are about to flip.

In [ ]:
from workshop_lib import run_tool_loop, TOOL_SCHEMAS, ocr_page

impls = {
    "get_page_text": lambda page: get_page_text(legacy, page),
    "ocr_page":      lambda page: ocr_page(legacy, page),
}
system = ("You are a taxonomic data extraction assistant reading scanned "
          "historical literature. The embedded text layer of a scan is often "
          "corrupt: watch for garbled words and impossible spellings of "
          "taxonomic names. If the text looks corrupt, re-read the page with "
          "ocr_page before trusting it.")
question = [{"role": "system", "content": system},
            {"role": "user", "content":
             "On page 5, what is the family name printed in the running header? "
             "Report it exactly."}]

answer, calls = run_tool_loop(question, TOOL_SCHEMAS, impls, think=False)
print("tools chosen:", calls)
print(answer)

With reasoning off, the loop calls `get_page_text`, gets the corrupt text
back, and answers from it anyway -- confidently, and often worse than the
text layer itself. Running this notebook, the model did not even reproduce
the text layer's own error faithfully: the text layer says
`Cureulionidse`, the page actually reads `Curculionidae`, and the model
answered with a *third* spelling, `Cureulionidae`, matching neither. It
noticed something was off -- you can sometimes see it hedge, calling the
garbling a minor transcription artifact -- and talked itself into trusting
the text anyway.

This was reproduced across four different system prompts, including much
more explicit warnings about corrupt scans than the one above. Prompting
did not fix it. A system prompt tells the model what to watch for; it does
not hand the model the reasoning capacity to act on what it noticed.

In [ ]:
answer, calls = run_tool_loop(question, TOOL_SCHEMAS, impls, think=True)
print("tools chosen:", calls)   # expect get_page_text THEN ocr_page
print(answer)                   # expect: Curculionidae

One argument changed. The loop now calls `get_page_text`, recognises the
result as untrustworthy, calls `ocr_page` on its own initiative, and
answers `Curculionidae` -- correctly -- in roughly a minute (it varies with
model load; this notebook's own run of the cell above took just under 62
seconds).

This is the payoff Section 5 asked you to hold onto. Transcription is pure
recall: there is nothing to reason about, so `think=True` there just burns
time producing nothing (recall the 123,055 characters of reasoning that
returned no answer at all). Here the task is different -- *is this page's
text trustworthy, and if not, what do I do about it* -- and that is
judgement, not recall. Reasoning is not a strictly-better setting you
should always leave on; it is the thing that makes this particular loop
work at all, because the loop's whole job is a decision the general model
has to think through.

The wording of the system prompt matters less than you might expect, once
reasoning is on. A much longer, more explicit version -- one that told the
model to literally count garbled-looking words before deciding -- took 186
seconds to reach the same answer the plain prompt above reached in about a
minute. More instruction is not obviously better; it can just be more for
the model to reason through on the way to the same place.

### Why there is no regex version of this

You may be thinking: why involve a language model at all? Page 5's text
layer contains a nonsense spelling -- couldn't a plain rule just flag "this
page looks corrupt" and skip the reasoning entirely?

An earlier draft of this workshop tried exactly that: a deterministic
`looks_corrupt()` function, no model involved, just pattern-matching on the
text layer. Getting it to work on nothing more than these *same two example
PDFs* required, in order: stripping URLs before scoring, because the
clean, born-digital cover page of the modern PDF is dense with DOIs and
punctuation that scored as *badly damaged* purely for containing them;
hand-tuning separate patterns for stray tildes and ampersands, because this
particular scanner's failure modes kept surfacing as those specific
characters; and then repairing a length/garbling ratio threshold that was
wrong in *both* directions at once -- it passed four genuinely corrupt
pages, one of them containing the string `Cureulioni&e.`, while
simultaneously flagging the one page that was completely clean.

Every one of those patches was a rule fitted to this scanner's particular
damage, on these two documents, discovered by looking at the answers first
and writing a regex to match them. There is no finite list of ways OCR can
be wrong. A rule tuned to catch `Cureulionidse` will not catch whatever a
different scanner, a different font, or a different century of paper
produces instead -- and when it misses, it does not raise an error or
return an empty string. It returns confident, plausible-looking prose, and
your pipeline sails on with a wrong family name in a database somewhere.
That is the worst way to fail: silently, and downstream of anyone who would
notice.

That leaves two honest options:

1. **You already know.** In real work you almost always know which of your
   PDFs are scans and which are born-digital -- so just OCR the scans. No
   detection needed, no cleverness, no rule to maintain. This covers most
   of what you will actually do with a stack of literature.
2. **Ask a model that can reason**, as in Stage 2 above, for the genuine
   edge case: a mixed pile where you do not know, and cannot easily check,
   which files are which.

Neither option is "write a rule to detect corruption." That option looks
attractive right up until you watch it fail on the one page it was not
tuned for.

### Hands-on 3 (capstone)

Run the Stage 2 loop -- reasoning on -- over every page of the legacy PDF
(pages 2 through 8; page 1 is a journal cover page, not article text) and
over two or three pages of the modern one. For each page, record which
tool(s) the model called and why. The modern PDF is born-digital -- watch
whether the model still checks every time, or learns page by page that it
does not need to. It has no memory between pages here, so any consistency
you see is the model reasoning its way to the same conclusion each time,
not remembering the last one.

In [ ]:
import time

def impls_for(doc):
    return {"get_page_text": lambda page: get_page_text(doc, page),
            "ocr_page":      lambda page: ocr_page(doc, page)}

header_question = ("On page {p}, what text is printed in the running header "
                    "at the top of the page? Report it exactly.")

jobs = [(legacy, "legacy", p) for p in range(2, 9)] + \
       [(modern, "modern", p) for p in range(2, 4)]

for doc, label, p in jobs:
    q = [{"role": "system", "content": system},
         {"role": "user", "content": header_question.format(p=p)}]
    t0 = time.time()
    answer, calls = run_tool_loop(q, TOOL_SCHEMAS, impls_for(doc), think=True)
    elapsed = time.time() - t0
    tools_used = [name for name, _ in calls]
    print(f"{label} p{p}  ({elapsed:4.1f}s)  tools={tools_used}")
    print(f"    -> {answer.strip()[:150]}")

Look at which pages triggered `ocr_page` and which did not. For the legacy
PDF, that split does not track something as simple as "even pages" or "odd
pages" -- it tracks where the model judged the text layer untrustworthy.
For the modern PDF, notice whether the model checked at all, or spent its
~50 extra seconds of reasoning on a page that never needed OCR in the first
place.

Discuss: for which of these pages would you rather have simply *known*
which file was a scan in advance, and skipped the model entirely? And for
which page did the extra reasoning time actually buy you something --
a correction -- you could not have gotten any cheaper?

## 8. The same code, in the cloud

Everything so far has run on a laptop. Ollama also hosts a small number of
larger models in the cloud, reachable through the exact same `ollama.chat`
calls -- only the model name and an API key change. This matters for
anything too large to fit in a workshop laptop's memory, and it costs
nothing to try if you already have the free account.

The open question this section is actually asking: the 9B model you have
been using needed `think=True` to escalate to OCR at all -- with reasoning
off it never once checked, no matter how the system prompt was worded.
Does a much larger cloud model still need that crutch, or can a bigger
model route correctly with reasoning off? Nobody at this workshop has
measured that. Run the cell below with `model=CLOUD_MODEL` both ways
(`think=True` and `think=False`) on your own account and find out -- this
is a genuinely open question, and exactly the kind of thing worth checking
on your own documents rather than assuming an answer either way.

Running this cell for real requires a free `ollama.com` account and an API
key added to Colab Secrets as `OLLAMA_API_KEY`. Without one, it prints
instructions instead of raising:

In [ ]:
# Same code, one different string. Requires a free ollama.com account;
# add OLLAMA_API_KEY to Colab Secrets first.
try:
    from google.colab import userdata
    import os
    os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

    CLOUD_MODEL = "qwen3.5:cloud"
    answer, calls = run_tool_loop(
        [{"role": "system", "content": system},
         {"role": "user", "content":
          "On page 5, what is the family name in the running header?"}],
        TOOL_SCHEMAS, impls, model=CLOUD_MODEL)
    print("tools the cloud model chose:", calls)
    print(answer)
except Exception as exc:
    print(f"Skipping the cloud demo here ({type(exc).__name__}: {exc}).")
    print("To run this for real: on Colab, add OLLAMA_API_KEY to Colab "
          "Secrets (a free ollama.com account is enough), then re-run this "
          "cell. Off Colab, `export OLLAMA_API_KEY=...` before starting "
          "Jupyter and drop the google.colab import above.")

## 9. Where this goes from here

### Portability: same server, different client

Ollama exposes an OpenAI-compatible endpoint, so none of this is locked to
the `ollama` python package. Point the standard `openai` client at your
local server instead, and the tool-calling and structured-output patterns
from Sections 6 and 7 carry over -- `.parse()` even handles turning a
pydantic model into the right JSON-Schema request for you:

In [ ]:
from openai import OpenAI
from workshop_lib import Extraction, EXTRACT_PROMPT

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Tools and schemas port cleanly.
parsed = client.chat.completions.parse(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": EXTRACT_PROMPT + clean[:4000]}],
    response_format=Extraction,
    temperature=0,
).choices[0].message.parsed
print(parsed.species[0].name)

That ran through a different client library without changing a single line
of the extraction logic. But "OpenAI-compatible" is not the same as
"identical," and the gap is worth seeing before you rely on it. Ask a
reasoning question through the same client and look at where the reasoning
actually ends up:

In [ ]:
reply = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content":
               "A beetle is 4.2 mm long and 1.4 mm wide. Ratio?"}],
    temperature=0)
msg = reply.choices[0].message
print("content:", msg.content)
print("reasoning_content present?", "reasoning_content" in (msg.model_extra or {}))
print("model_extra keys:", list((msg.model_extra or {}).keys()))

Tool calling and structured output ported cleanly across the two clients --
that part of the abstraction holds. Reasoning does not: it comes back under
a bare `reasoning` key inside `model_extra`, which is neither the
`reasoning_content` field used by DeepSeek and vLLM nor anything defined by
the OpenAI API itself. The typed SDK has no field for it, so code written
against `msg.content` alone will not even notice that a chain of thought is
sitting one level down in an untyped dict. That is precisely the kind of
thing that breaks quietly when you swap providers next year -- not the
part you tested, but the part you assumed was standard because it happened
to work once.

### Closing the loop

Three practical points worth carrying out of this room:

- **Reproducibility.** If you archive weights alongside a methods section,
  pin the exact model tag (`qwen3.5:9b`, not "qwen") *and* the Ollama
  version you ran it on -- these particular models will not even load on
  Ollama 0.13.2. "Open weights" only buys you reproducibility if you also
  freeze the software that reads them.
- **Data sovereignty.** Unpublished specimen records and localities for
  threatened taxa never have to leave your machine. That is not true of
  any hosted API, however good its privacy policy reads.
- **Zero marginal cost.** Once the weights are on disk, the fiftieth
  extraction costs exactly what the first one did: nothing. That changes
  how you use these tools -- you can afford to be wrong, re-run, and
  iterate on a prompt or a schema all afternoon, in a way that metered API
  calls actively discourage.

None of this replaces checking the output against a specimen you can put
your hand on. It just means the checking is the only expensive part
left.